# LAB01 — SVM guiada (práctica metodológica)

**INF-8239 Ciencia de Datos II · Unidad 01 · Ejercicio 01**

Este notebook practica la metodología completa (split → baseline → pipeline SVM → GridSearchCV → métricas) usando el dataset `load_breast_cancer` de scikit-learn.

> ⚠️ **Advertencia académica:** Este experimento tiene finalidad exclusivamente educativa y metodológica. Los resultados no constituyen una herramienta médica ni deben utilizarse para diagnóstico o decisiones clínicas.

Este dataset es **solo práctica guiada**; el dataset final del Ejercicio 01 se selecciona y audita en el LAB02 (`02_dataset_auditoria.ipynb`).

## 1. Carga del dataset de práctica

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X = data.frame.drop(columns='target')
y = data.frame['target']

print('Muestras:', X.shape[0])
print('Variables predictoras:', X.shape[1])
print('Clases:', sorted(y.unique().tolist()))
print('Distribución de clases:')
print(y.value_counts())

Muestras: 569
Variables predictoras: 30
Clases: [0, 1]
Distribución de clases:
target
1    357
0    212
Name: count, dtype: int64


## 2. Split estratificado train/test

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Distribución train:', y_train.value_counts().to_dict())
print('Distribución test:', y_test.value_counts().to_dict())

Train: (455, 30) | Test: (114, 30)
Distribución train: {1: 285, 0: 170}
Distribución test: {1: 72, 0: 42}


## 3. Baseline — DummyClassifier

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)

print('Accuracy baseline:', round(accuracy_score(y_test, pred_base), 4))
print('F1 macro baseline:', round(f1_score(y_test, pred_base, average='macro'), 4))

Accuracy baseline: 0.6316
F1 macro baseline: 0.3871


## 4. Pipeline SVM (StandardScaler → SVC)

El escalado vive **dentro** del pipeline: nunca se escala el dataset completo antes del split (prevención de data leakage).

In [4]:
from inf8239_u01.models import build_svm

svm = build_svm(C=1.0, gamma='scale')
print(svm)
svm.fit(X_train, y_train)
pred_svm = svm.predict(X_test)

print('Accuracy SVM:', round(accuracy_score(y_test, pred_svm), 4))
print('F1 macro SVM:', round(f1_score(y_test, pred_svm, average='macro'), 4))

Pipeline(steps=[('scale', StandardScaler()),
                ('model', SVC(probability=True, random_state=42))])
Accuracy SVM: 0.9825
F1 macro SVM: 0.9812


## 5. GridSearchCV (SOLO sobre entrenamiento)

In [5]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 0.01, 0.1],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=build_svm(),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print('Mejores parámetros:', grid.best_params_)
print('Mejor F1 macro (CV):', round(grid.best_score_, 4))

Mejores parámetros: {'model__C': 10, 'model__gamma': 0.01}
Mejor F1 macro (CV): 0.9739


## 6. Métricas finales sobre el conjunto de test

In [6]:
from sklearn.metrics import classification_report, roc_auc_score

best = grid.best_estimator_
pred_best = best.predict(X_test)
proba_best = best.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred_best))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_best), 4))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

ROC-AUC: 0.9977


## Conclusión del LAB01

Se practicó el flujo metodológico completo con split estratificado, baseline `DummyClassifier`, pipeline SVM con escalado interno y búsqueda de hiperparámetros mediante validación cruzada estratificada. Esta misma estructura se aplica al dataset público seleccionado en el LAB02.